# Task 2 — Imaging Model (MONAI CNN)
**COMP41840 AI for Health**  
**Owner:** Thomas  
**Goal:** Binary classification — benign vs malignant — from ultrasound images using a MONAI DenseNet121


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

import monai
from monai.networks.nets import DenseNet121
from monai.transforms import (
    Compose, LoadImage, EnsureChannelFirst, ScaleIntensity,
    Resize, RandFlip, RandRotate, ToTensor
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

DATA_ROOT = Path('../data/raw')
RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)
(RESULTS_DIR / 'figures').mkdir(exist_ok=True)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## 2.1 — Build Image/Label Lists (benign=0, malignant=1)

In [ ]:
# Only benign and malignant for binary classification
image_paths, labels = [], []

for cls, label in [('benign', 0), ('malignant', 1)]:
    img_dir = DATA_ROOT / cls / 'images'
    paths = sorted(img_dir.glob('*.png'))
    image_paths.extend(paths)
    labels.extend([label] * len(paths))

print(f'Total images: {len(image_paths)}')
print(f'Benign: {labels.count(0)}  |  Malignant: {labels.count(1)}')

# Stratified train/val/test split: 70/15/15
X_train, X_temp, y_train, y_temp = train_test_split(
    image_paths, labels, test_size=0.3, stratify=labels, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED
)

print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')

## 2.2 — Dataset & DataLoaders

In [ ]:
class BreastUltrasoundDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

train_ds = BreastUltrasoundDataset(X_train, y_train, train_transform)
val_ds   = BreastUltrasoundDataset(X_val,   y_val,   val_transform)
test_ds  = BreastUltrasoundDataset(X_test,  y_test,  val_transform)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False, num_workers=2)

## 2.3 — Model: MONAI DenseNet121

In [ ]:
model = DenseNet121(
    spatial_dims=2,
    in_channels=3,
    out_channels=2  # benign / malignant
).to(DEVICE)

# Class weights to handle imbalance
n_benign    = labels.count(0)
n_malignant = labels.count(1)
weights = torch.tensor([1.0, n_benign / n_malignant]).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

print(f'Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 2.4 — Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct = 0.0, 0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == lbls).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)


def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            out = model(imgs)
            loss = criterion(out, lbls)
            total_loss += loss.item() * imgs.size(0)
            correct += (out.argmax(1) == lbls).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)


EPOCHS = 30
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    vl_loss, vl_acc = eval_epoch(model,  val_loader,   criterion, DEVICE)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        torch.save(model.state_dict(), RESULTS_DIR / 'best_imaging_model.pt')

    print(f'Epoch {epoch+1:02d}/{EPOCHS} | '
          f'Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | '
          f'Val Loss: {vl_loss:.4f} Acc: {vl_acc:.4f}')

## 2.5 — Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'],   label='Val')
ax1.set_title('Loss'); ax1.legend()
ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'],   label='Val')
ax2.set_title('Accuracy'); ax2.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures/imaging_training_curves.png', dpi=150)
plt.show()

## 2.6 — Test Set Evaluation

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(RESULTS_DIR / 'best_imaging_model.pt', map_location=DEVICE))
model.eval()

all_preds, all_probs, all_labels = [], [], []
with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs = imgs.to(DEVICE)
        out  = model(imgs)
        probs = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
        preds = out.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_probs.extend(probs)
        all_labels.extend(lbls.numpy())

print(classification_report(all_labels, all_preds, target_names=['benign', 'malignant']))
print(f'AUC-ROC: {roc_auc_score(all_labels, all_probs):.4f}')

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['benign', 'malignant'])
disp.plot(cmap='Blues')
plt.title('Imaging Model — Confusion Matrix')
plt.savefig(RESULTS_DIR / 'figures/imaging_confusion_matrix.png', dpi=150)
plt.show()

# ROC Curve
RocCurveDisplay.from_predictions(all_labels, all_probs, name='DenseNet121')
plt.title('Imaging Model — ROC Curve')
plt.savefig(RESULTS_DIR / 'figures/imaging_roc.png', dpi=150)
plt.show()

In [ ]:
# Save metrics for fusion notebook
import json
from sklearn.metrics import precision_score, recall_score, f1_score

imaging_metrics = {
    'auc': round(roc_auc_score(all_labels, all_probs), 4),
    'precision': round(precision_score(all_labels, all_preds), 4),
    'recall': round(recall_score(all_labels, all_preds), 4),
    'f1': round(f1_score(all_labels, all_preds), 4),
    'probs': all_probs,   # needed by fusion notebook
    'preds': all_preds
}

# Save probs/preds for fusion
np.save(RESULTS_DIR / 'imaging_test_probs.npy', np.array(all_probs))
np.save(RESULTS_DIR / 'imaging_test_labels.npy', np.array(all_labels))

metrics_path = RESULTS_DIR / 'metrics.json'
existing = json.loads(metrics_path.read_text()) if metrics_path.exists() else {}
existing['imaging'] = {k: v for k, v in imaging_metrics.items() if k not in ['probs', 'preds']}
metrics_path.write_text(json.dumps(existing, indent=2))
print('Saved:', imaging_metrics)